# Analisis Dialog *Byeolisam*

Notebook awal untuk menjelajah dataset: siapa yang paling banyak bicara, gaya bicara, kata khas, dan interaksi antar tokoh.
Jalankan sel dari atas ke bawah. Arti kolom ada di `README.md`.

In [ ]:
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

from matplotlib import font_manager
_ada = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ("Malgun Gothic", "AppleGothic", "NanumGothic") if f in _ada), "DejaVu Sans")  # huruf Korea
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_colwidth", 80)

df = pd.read_csv("data/dialog.csv", encoding="utf-8-sig")
tokoh = pd.read_csv("data/tokoh.csv", encoding="utf-8-sig")
NAMA = dict(zip(tokoh.pembicara, tokoh.pembicara_romawi))   # 지수원 -> Ji Suwon
BUKAN = {"Unknown", "Narration", "SFX", "Komik dalam komik"}
print(len(df), "teks,", df.chapter.nunique(), "chapter")
df.head()

## 1. Daftar pembicara

In [ ]:
tokoh

## 2. Kemunculan tokoh per chapter

In [ ]:
utama = tokoh[tokoh.jenis == "tokoh bernama"].head(12).pembicara
tabel = (df[df.pembicara.isin(utama)].groupby(["pembicara", "chapter"]).size().unstack(fill_value=0)
         .reindex(utama))
fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(tabel.values, aspect="auto", cmap="Blues")
ax.set_yticks(range(len(tabel)), [NAMA[n] for n in tabel.index])
ax.set_xticks(range(0, tabel.shape[1], 5), tabel.columns[::5])
ax.set_title("Jumlah teks per chapter (makin gelap makin banyak)")
plt.show()

## 3. Bicara vs. berpikir
Tokoh dengan porsi `pikiran` besar biasanya tokoh sudut pandang (POV).

In [ ]:
t = tokoh[tokoh.jenis == "tokoh bernama"].head(15).set_index("pembicara_romawi")
(t[["ucapan", "pikiran", "tulisan", "ingatan"]].div(t.jumlah_teks, axis=0)
 .plot.barh(stacked=True, figsize=(9, 6), title="Proporsi tipe teks")).invert_yaxis()
plt.show()

## 4. Gaya bicara
- **sopan** = kalimat berakhiran 요 / 니다 / 니까 (존댓말)
- **ㅋ** = tawa internet, **ㅅㅂ** dll. = makian

In [ ]:
SOPAN = re.compile(r"(?:요|니다|니까|세요|죠)[.!?~…\s]*$")
MAKI = re.compile(r"ㅅㅂ|씨발|시발|ㅂㅅ|병신|새끼|존나|좆")

def gaya(g):
    t = g.teks.astype(str)
    return pd.Series({"teks": len(t), "panjang_rata2": t.str.len().mean(),
                      "% sopan": 100 * t.str.contains(SOPAN).mean(), "% tanya": 100 * t.str.contains(r"\?").mean(),
                      "% seru": 100 * t.str.contains("!").mean(), "% ㅋ": 100 * t.str.contains("ㅋ").mean(),
                      "% makian": 100 * t.str.contains(MAKI).mean()})

ucap = df[df.tipe.isin(["ucapan", "tulisan"]) & ~df.pembicara.isin(BUKAN)]
g = ucap.groupby("pembicara").apply(gaya, include_groups=False)
g = g[g.teks >= 30].sort_values("teks", ascending=False)
g.index = [NAMA.get(n, n) for n in g.index]
g.round(1)

## 5. Kata khas tiap tokoh
Pakai `kiwipiepy` untuk memecah kalimat Korea menjadi kata dasar. Skor = seberapa sering tokoh itu memakai kata tersebut
dibanding tokoh lain (log-odds sederhana), jadi kata umum seperti 나/너 tidak mendominasi.

In [ ]:
import math
from kiwipiepy import Kiwi
kiwi = Kiwi()
AMBIL = {"NNG", "NNP", "VV", "VA", "MAG", "IC", "XR"}   # kata benda, kerja, sifat, keterangan, seruan

def kata(teks):
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(str(teks)) if t.tag in AMBIL]

ucap = ucap.assign(kata=ucap.teks.map(kata))
semua = Counter(w for ws in ucap.kata for w in ws)
N = sum(semua.values())

def kata_khas(nama, top=15, min_n=5):
    c = Counter(w for ws in ucap[ucap.pembicara == nama].kata for w in ws)
    n = sum(c.values())
    skor = {w: math.log((k + 1) / (n + 1)) - math.log((semua[w] - k + 1) / (N - n + 1)) for w, k in c.items() if k >= min_n}
    return pd.DataFrame(sorted(skor.items(), key=lambda x: -x[1])[:top], columns=["kata", "skor"]).assign(
        jumlah=lambda d: d.kata.map(c))

for nama in tokoh[tokoh.jenis == "tokoh bernama"].pembicara[:4]:
    print("==", nama, NAMA[nama]); print(kata_khas(nama).to_string(index=False))

## 6. Siapa bicara dengan siapa
Hitungan sederhana: dua ucapan berurutan dari tokoh berbeda di chapter yang sama dan panel berdekatan dianggap satu
pertukaran. (Urutan bubble yang sejajar belum dicek penuh — lihat README.)

In [ ]:
u = df[(df.tipe == "ucapan") & ~df.pembicara.isin(BUKAN)].sort_values("urutan")
pasangan = Counter()
for (a, b) in zip(u.itertuples(), u.iloc[1:].itertuples()):
    if a.chapter == b.chapter and b.panel - a.panel <= 1 and a.pembicara != b.pembicara:
        pasangan[tuple(sorted((NAMA.get(a.pembicara, a.pembicara), NAMA.get(b.pembicara, b.pembicara))))] += 1
pd.DataFrame([(x, y, n) for (x, y), n in pasangan.most_common(20)], columns=["tokoh A", "tokoh B", "pertukaran"])

## 7. Alat bantu: cari teks dan lihat adegan

In [ ]:
def cari(pola, pembicara=None, tipe=None):
    """cari('사랑'), cari('ㅋㅋ', pembicara='박동'), cari(r'형[.!?]', tipe='ucapan')"""
    d = df[df.teks.astype(str).str.contains(pola, regex=True)]
    if pembicara: d = d[d.pembicara == pembicara]
    if tipe: d = d[d.tipe == tipe]
    return d[["urutan", "chapter", "pembicara_romawi", "tipe", "teks"]]

def adegan(urutan, n=6):
    """tampilkan n teks sebelum dan sesudah nomor `urutan` (kolom urutan)"""
    return df[df.urutan.between(urutan - n, urutan + n)][["urutan", "chapter", "pembicara_romawi", "tipe", "teks"]]

cari("사랑").head(10)

In [ ]:
adegan(100)

In [ ]:
from pathlib import Path
from matplotlib.patches import Rectangle

def lihat(bubble_id, tinggi=9):
    """tampilkan potongan panel tempat teks itu berada; kotak merah = teks yang dicari, biru = teks lain di panel itu"""
    r = df[df.bubble_id == bubble_id].iloc[0]
    if not Path(r.gambar).exists():
        return print("gambar belum ada:", r.gambar, "(folder panels/ tidak ditemukan)")
    im = plt.imread(r.gambar)
    fig, ax = plt.subplots(figsize=(tinggi * im.shape[1] / im.shape[0] + 3, tinggi))
    ax.imshow(im); ax.axis("off")
    for s in df[df.gambar == r.gambar].itertuples():
        x1, y1, x2, y2 = map(int, s.kotak.split(","))
        warna = "red" if s.bubble_id == bubble_id else "royalblue"
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=warna, lw=2))
        ax.text(x2 + 4, y1, f"{s.urutan} {s.pembicara_romawi} ({s.tipe})", color=warna, fontsize=8, va="top",
                bbox=dict(fc="white", alpha=0.8, lw=0))
    ax.set_title(f"{r.chapter}  {r.gambar}  posisi {r.posisi_persen}% chapter")
    plt.show()
    return df[df.gambar == r.gambar][["urutan", "pembicara_romawi", "tipe", "teks"]]

lihat(df.bubble_id[100])